# Early stopping

_Deep Learning_

**Stop training the moment the validation error stops improving.**

Train too long and the network starts memorizing the training set. That is overfitting.

     Early stopping watches a separate validation set (data the model isn't trained on).

---

This notebook is a practice scaffold for the **Early stopping** lesson. We build it up one small step at a time: run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PyTorch

### Step 1 — Set up the model, data, and patience counter

We make a tiny linear model with an Adam optimizer and an MSE loss, plus random train and validation tensors to stand in for real data. The early-stopping bookkeeping lives in four variables: `best_loss` (the lowest validation loss seen so far), `best_state` (a snapshot of the weights at that point), `patience` (how many epochs of no improvement we tolerate), and `wait` (epochs since the last improvement).

In [ ]:
import torch
import torch.nn as nn
import copy

model = nn.Linear(10, 1)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
Xtr, ytr = torch.randn(200, 10), torch.randn(200, 1)
Xval, yval = torch.randn(50, 10), torch.randn(50, 1)

best_loss = float("inf")
best_state = None
patience = 5
wait = 0

### Step 2 — Train, watching validation loss for early stopping

Each epoch we take one gradient step on the training data, then measure the loss on the held-out validation set. If validation improved, we record the new best loss, snapshot the weights, and reset `wait` to 0. If it didn't, we increment `wait`; once it reaches `patience` (5 epochs with no improvement) we stop the loop early instead of training all 100 epochs.

In [ ]:
for epoch in range(100):
    opt.zero_grad()
    train_loss = loss_fn(model(Xtr), ytr)
    train_loss.backward()
    opt.step()

    with torch.no_grad():
        val = loss_fn(model(Xval), yval).item()   # check validation

    if val < best_loss:
        best_loss = val
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:                       # no improvement for 5 epochs
            print("stopped at epoch", epoch, "best val", round(best_loss, 4))
            break

### Step 3 — Restore the best weights

When the loop stops, the model's current weights are from the *last* (worse) epochs, not the best one. Early stopping's final move is to load back the snapshot we saved at the lowest validation loss, so the model we keep is the best one we ever saw — not the one we happened to end on.

In [ ]:
model.load_state_dict(best_state)                  # restore best weights

## Visualize the data & results

_Training on real tumor data, when does validation loss bottom out and start rising?_

### Step 1 — Load and split the real tumor data

We load the breast-cancer dataset, standardise every feature, and split it into a training set and a held-out test set that acts as our validation set. The split is **stratified** so both sets keep the same balance of malignant and benign cases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)
Xtr, Xte, ytr, yte = train_test_split(
    X, bc.target, test_size=0.3, random_state=0, stratify=bc.target
)

### Step 2 — Train epoch by epoch, recording both losses

We train an MLP one epoch at a time (`warm_start` + `partial_fit`) for 40 epochs, recording the train and validation log-loss after each. Tracking both curves is exactly what early stopping needs: the training loss keeps falling, but the validation loss is the one that tells us when to stop.

In [ ]:
m = MLPClassifier(hidden_layer_sizes=(64, 64), solver="adam", max_iter=1,
                  warm_start=True, random_state=0, alpha=1e-5,
                  learning_rate_init=0.003)
tr, va = [], []
for e in range(40):
    if e == 0:
        m.partial_fit(Xtr, ytr, classes=[0, 1])
    else:
        m.partial_fit(Xtr, ytr)
    tr.append(log_loss(ytr, m.predict_proba(Xtr), labels=[0, 1]))
    va.append(log_loss(yte, m.predict_proba(Xte), labels=[0, 1]))

### Step 3 — Find and plot the early-stopping point

The best epoch is simply the one with the lowest validation loss — `np.argmin(va)`. We draw the train and validation curves and mark that epoch with a dashed line. To the right of it the validation loss rises again: training past that point only overfits, which is exactly what early stopping prevents.

In [ ]:
stop = int(np.argmin(va))               # real early-stopping epoch = 12
plt.plot(tr, color="#4ea1ff", label="train loss")
plt.plot(va, color="#ffb454", label="validation loss")
plt.axvline(stop, color="gray", linestyle="--", label="early stop")
plt.title("Real breast-cancer training: validation bottoms out at epoch 12")
plt.xlabel("epoch")
plt.ylabel("log loss")
plt.legend()
plt.show()